In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multioutput import MultiOutputClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform
import re
from nltk.stem import WordNetLemmatizer
from joblib import parallel_backend

# --- CARGA DE DATOS ---
train_data = pd.read_csv("../../Data/train_indexado.csv")
test_data = pd.read_csv("../../Data/test_indexado.csv")
emotion_classes = train_data.columns[2:].tolist()

# --- PREPROCESAMIENTO (Lematización) ---
lemmatizer = WordNetLemmatizer()
def preprocess_text(text):
    words = re.findall(r'\b\w+\b', text.lower())
    return " ".join(lemmatizer.lemmatize(w) for w in words)

X_train_lem = train_data['Text'].apply(preprocess_text)
X_test_lem = test_data['Text'].apply(preprocess_text)

# --- TF-IDF ---
vectorizer = TfidfVectorizer(
    lowercase=True, 
    strip_accents="unicode", 
    max_features=8000,   # 🔽 reducir un poco mejora tiempo sin afectar mucho performance
    ngram_range=(1,2),   # ⚡ bigramas para mejor contexto
    sublinear_tf=True
)
X_train = vectorizer.fit_transform(X_train_lem)
X_test = vectorizer.transform(X_test_lem)

y_train = np.asarray(train_data[emotion_classes])
y_test = np.asarray(test_data[emotion_classes])

# --- FEATURE SELECTION (Chi²) ---
selector = SelectKBest(chi2, k=1000)
X_train_chi = selector.fit_transform(X_train, y_train)
X_test_chi = selector.transform(X_test)

print(f"Datos preparados: {X_train_chi.shape[0]} muestras, {X_train_chi.shape[1]} features seleccionadas")


Datos preparados: 43410 muestras, 1000 features seleccionadas


In [2]:
# --- MLP BASE ---
mlp_base = MLPClassifier(
    random_state=42,
    early_stopping=True,
    n_iter_no_change=10,
    max_iter=500,        
    learning_rate_init=0.001,
)

multi_mlp = MultiOutputClassifier(mlp_base, n_jobs=-1)

# --- HIPERPARÁMETROS (RandomizedSearchCV más eficiente) ---
param_distributions = {
    "estimator__hidden_layer_sizes": [(100,), (200,), (100, 50)],
    "estimator__activation": ["relu", "tanh"],
    "estimator__alpha": loguniform(1e-5, 1e-2),
    "estimator__learning_rate": ["constant", "adaptive"],
}

search = RandomizedSearchCV(
    multi_mlp,
    param_distributions=param_distributions,
    n_iter=10,            # 🔽 solo 10 combinaciones aleatorias
    scoring="f1_macro",
    cv=3,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

print("Iniciando búsqueda aleatoria de hiperparámetros...")
with parallel_backend('loky'):
    search.fit(X_train_chi, y_train)

print("\n=== MEJORES PARÁMETROS ENCONTRADOS ===")
print(search.best_params_)
print(f"Mejor F1 (CV): {search.best_score_:.4f}")

# --- EVALUACIÓN FINAL ---
best_model = search.best_estimator_
y_pred = best_model.predict(X_test_chi)

report = classification_report(y_test, y_pred, zero_division=0)
f1_macro = classification_report(y_test, y_pred, output_dict=True, zero_division=0)["macro avg"]["f1-score"]

print("\n=== RESULTADOS FINALES ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 macro: {f1_macro:.4f}")
print("\nReporte completo:\n", report)

Iniciando búsqueda aleatoria de hiperparámetros...
Fitting 3 folds for each of 10 candidates, totalling 30 fits

=== MEJORES PARÁMETROS ENCONTRADOS ===
{'estimator__activation': 'relu', 'estimator__alpha': 0.0024526126311336773, 'estimator__hidden_layer_sizes': (100, 50), 'estimator__learning_rate': 'constant'}
Mejor F1 (CV): 0.3459

=== RESULTADOS FINALES ===
Accuracy: 0.3820
F1 macro: 0.3735

Reporte completo:
               precision    recall  f1-score   support

           0       0.68      0.49      0.57       504
           1       0.79      0.75      0.77       264
           2       0.63      0.21      0.31       198
           3       0.62      0.09      0.15       320
           4       0.65      0.15      0.24       351
           5       0.54      0.11      0.18       135
           6       0.58      0.14      0.23       153
           7       0.51      0.20      0.29       284
           8       0.75      0.18      0.29        83
           9       0.67      0.08      0.1